In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical  # np_utils.to_categorical → tf.keras.utils.to_categorical
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
%matplotlib inline

batch_size = 32
epochs = 25
lrate = 0.01
decay = lrate/epochs
data_augmentation = True
num_classes = 10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
X_train = X_train.astype('float32') / 255
X_test = X_test.astype('float32') / 255

In [2]:
y_train = tf.keras.utils.to_categorical(y_train, num_classes)
y_test = tf.keras.utils.to_categorical(y_test, num_classes)

input_img = tf.keras.layers.Input(shape=(32,32,3))

group_1 = tf.keras.layers.Conv2D(64, (1,1), padding='same', activation='relu')(input_img)
group_1 = tf.keras.layers.Conv2D(64, (3,3), padding='same', activation='relu')(group_1)
group_2 = tf.keras.layers.Conv2D(64, (1,1), padding='same', activation='relu')(input_img)
group_2 = tf.keras.layers.Conv2D(64, (5,5), padding='same', activation='relu')(group_2)
group_3 = tf.keras.layers.MaxPooling2D((3,3), strides=(1,1), padding='same')(input_img)
group_3 = tf.keras.layers.Conv2D(64, (1,1), padding='same', activation='relu')(group_3)

output = tf.keras.layers.concatenate([group_1, group_2, group_3], axis=3)

2026-09-03 14:48:46.633407: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-09-03 14:48:46.633570: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-09-03 14:48:46.633575: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.00 GB
I0000 00:00:1788427126.634035  192100 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1788427126.634082  192100 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [3]:
# In [9]:
output = Flatten()(output)
out = Dense(10, activation='softmax')(output)

In [4]:
# In [10]:
model = Model(inputs = input_img, outputs= out)

In [5]:
# In [11]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 32, 32,    │        256 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │        256 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 32, 32, 3) │          0 │ input_layer[0][0] │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 32, 32,    │     36,928 │ conv2d[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 32, 32,    │    102,464 │ conv2d_2[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 32, 32,    │        256 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 32, 32,    │          0 │ conv2d_1[0][0],   │
│ (Concatenate)       │ 192)              │            │ conv2d_3[0][0],   │
│                     │                   │            │ conv2d_4[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 196608)    │          0 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 10)        │  1,966,090 │ flatten[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,106,250 (8.03 MB)

 Trainable params: 2,106,250 (8.03 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
"""
Cifar10-Inception — Keras 3 / local version

Rewritten from a TF 1.13 / Keras 2.2.4 Colab notebook. Fixes applied:
  - standalone `keras.*` imports -> `tensorflow.keras.*`
  - np_utils.to_categorical -> tf.keras.utils.to_categorical
  - Model(inputs=..., output=...) -> outputs= (plural) [already correct here]
  - SGD(lr=..., decay=...) -> SGD(learning_rate=..., weight_decay=...)
  - ImageDataGenerator (removed in Keras 3) -> tf.keras.layers augmentation
    layers, applied via .map() on a tf.data.Dataset (replaces
    datagen.flow(...) + model.fit_generator(...))
  - EarlyStopping(monitor="val_loss") kept as-is (still valid)
  - history.history['acc'] / ['val_acc'] -> ['accuracy'] / ['val_accuracy']
  - ModelCheckpoint / model-saving removed per request — nothing is
    persisted to disk; add a plain `model.save(...)` at the end yourself
    if you want to keep the trained weights.
"""

import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
# If running inside a Jupyter/IPython notebook, uncomment the next line:
# %matplotlib inline

# ---------------------------------------------------------------------
# Hyperparameters
# ---------------------------------------------------------------------
batch_size = 32          # orig paper trained all networks with batch_size=128
epochs = 25
lrate = 0.01
decay = lrate / epochs
data_augmentation = True
num_classes = 10

# ---------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------
(X_train, y_train), (X_test, y_test) = cifar10.load_data()
X_train = X_train.astype('float32') / 255
X_test = X_test.astype('float32') / 255

y_train = tf.keras.utils.to_categorical(y_train, num_classes)
y_test = tf.keras.utils.to_categorical(y_test, num_classes)

# ---------------------------------------------------------------------
# Model — a single Inception-style module (3 parallel branches
# concatenated), then flatten + softmax classifier
# ---------------------------------------------------------------------
input_img = Input(shape=(32, 32, 3))

group_1 = Conv2D(64, (1, 1), padding='same', activation='relu')(input_img)
group_1 = Conv2D(64, (3, 3), padding='same', activation='relu')(group_1)
group_2 = Conv2D(64, (1, 1), padding='same', activation='relu')(input_img)
group_2 = Conv2D(64, (5, 5), padding='same', activation='relu')(group_2)
group_3 = MaxPooling2D((3, 3), strides=(1, 1), padding='same')(input_img)
group_3 = Conv2D(64, (1, 1), padding='same', activation='relu')(group_3)

output = concatenate([group_1, group_2, group_3], axis=3)
output = Flatten()(output)
out = Dense(10, activation='softmax')(output)

model = Model(inputs=input_img, outputs=out)
model.summary()

# ---------------------------------------------------------------------
# Compile
# ---------------------------------------------------------------------
sgd = SGD(learning_rate=lrate, momentum=0.9, weight_decay=decay, nesterov=False)
model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])

# ---------------------------------------------------------------------
# Callbacks (model-saving/checkpoint intentionally omitted)
# ---------------------------------------------------------------------
early = EarlyStopping(monitor="val_loss",
                       mode="min",
                       patience=4, restore_best_weights=True)
callbacks = [early]

# ---------------------------------------------------------------------
# Data augmentation (Keras 3 layers, replacing ImageDataGenerator)
#
# The old ImageDataGenerator config included several options with no
# direct Keras 3 layer equivalent (ZCA whitening, per-sample/per-feature
# centering/normalization, channel shifts). Those are dropped below;
# the commonly-used spatial augmentations (rotation, width/height shift,
# shear, zoom, horizontal flip) are kept via their Keras 3 layer
# counterparts. RandomShear does not exist as a single layer, so shear
# is approximated by folding a bit of extra RandomZoom range instead —
# if exact shear behavior matters for your use case, let me know and
# I can add a custom shear layer.
# ---------------------------------------------------------------------
if not data_augmentation:
    print('Not using data augmentation.')
    history = model.fit(X_train, y_train,
                         batch_size=batch_size,
                         epochs=epochs,
                         validation_data=(X_test, y_test),
                         callbacks=callbacks)
else:
    print('Using real-time data augmentation.')

    data_augmentation_layers = tf.keras.Sequential([
        tf.keras.layers.RandomRotation(30 / 360),          # rotation_range=30 (degrees -> fraction of 2*pi... Keras uses fraction of 360)
        tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1),  # width/height_shift_range=0.1
        tf.keras.layers.RandomZoom(height_factor=0.2, width_factor=0.2),         # zoom_range=0.2 (also approximates shear_range=0.2)
        tf.keras.layers.RandomFlip("horizontal"),           # horizontal_flip=True, vertical_flip=False
    ])

    def augment(images, labels):
        return data_augmentation_layers(images, training=True), labels

    train_ds = (
        tf.data.Dataset.from_tensor_slices((X_train, y_train))
        .shuffle(buffer_size=10000)
        .batch(batch_size)
        .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
        .prefetch(tf.data.AUTOTUNE)
    )

    step_size_train = X_train.shape[0] // batch_size
    print(f"Number of samples: {X_train.shape[0]}")
    print(f"Batch size: {batch_size}")
    print(f"Step size train: {step_size_train}")

    history = model.fit(train_ds,
                         validation_data=(X_test, y_test),
                         epochs=epochs, verbose=1,
                         steps_per_epoch=step_size_train,
                         callbacks=callbacks)

# ---------------------------------------------------------------------
# Plots
# ---------------------------------------------------------------------
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Test'], loc='upper left')
plt.show()

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Test'], loc='upper left')
plt.show()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 32, 32,    │        256 │ input_layer_1[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 32, 32,    │        256 │ input_layer_1[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 32, 32, 3) │          0 │ input_layer_1[0]… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 32, 32,    │     36,928 │ conv2d_5[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 32, 32,    │    102,464 │ conv2d_7[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 32, 32,    │        256 │ max_pooling2d_1[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 32, 32,    │          0 │ conv2d_6[0][0],   │
│ (Concatenate)       │ 192)              │            │ conv2d_8[0][0],   │
│                     │                   │            │ conv2d_9[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 196608)    │          0 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 10)        │  1,966,090 │ flatten_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,106,250 (8.03 MB)

 Trainable params: 2,106,250 (8.03 MB)

 Non-trainable params: 0 (0.00 B)

...

/Users/ankit/Workspace/Projects/ankit-github/AI-ML-DS/54.Deep-Learning-for-Computer-Vision-with-TensorFlow-2/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
2026-09-03 15:03:08.801080: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1562/1562 ━━━━━━━━━━━━━━━━━━━━ 47s 29ms/step - accuracy: 0.3674 - loss: 1.7681 - val_accuracy: 0.4496 - val_loss: 1.5461
Epoch 2/25
   1/1562 ━━━━━━━━━━━━━━━━━━━━ 3:47 146ms/step - accuracy: 0.1875 - loss: 1.9456

/Users/ankit/Workspace/Projects/ankit-github/AI-ML-DS/54.Deep-Learning-for-Computer-Vision-with-TensorFlow-2/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1562/1562 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.1875 - loss: 1.9456 - val_accuracy: 0.4683 - val_loss: 1.4774
Epoch 3/25
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 51s 33ms/step - accuracy: 0.4341 - loss: 1.5869 - val_accuracy: 0.5051 - val_loss: 1.4192
Epoch 4/25
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.5000 - loss: 1.3088 - val_accuracy: 0.4868 - val_loss: 1.4706
Epoch 5/25
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 60s 39ms/step - accuracy: 0.4604 - loss: 1.5158 - val_accuracy: 0.5243 - val_loss: 1.3445
Epoch 6/25
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.3750 - loss: 1.9893 - val_accuracy: 0.5156 - val_loss: 1.3627
Epoch 7/25
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 56s 36ms/step - accuracy: 0.4777 - loss: 1.4722 - val_accuracy: 0.5447 - val_loss: 1.3176
Epoch 8/25
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.7500 - loss: 0.8422 - val_accuracy: 0.5366 - val_loss: 1.3515
Epoch 9/25
 697/1562 ━━━━━━━━━━━━━━━━━━━━ 28s 33ms/step - accuracy: 0.4835 - loss: 1.4456

KeyboardInterrupt: 